### PushT 1000-episode sim with Random Policy

In [3]:
import gymnasium as gym
import gym_pusht
import numpy as np 

env = gym.make("gym_pusht/PushT-v0", render_mode="rgb_array")

objc[32161]: Class AVFFrameReceiver is implemented in both /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x11be683a8) and /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x14c3f43a8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[32161]: Class AVFAudioReceiver is implemented in both /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x11be683f8) and /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/cv2/.dylibs/libavdevice.61.3.100.dylib (0x14c3f43f8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[32161]: Class SDLApplication is implemented in both /Users/snudurup

## 1. The Random policy

In [ ]:
success_lengths = []   # step counts of episodes that ended in success
failure_lengths = []   # step counts of episodes that hit the 300-step limit

num_ep = 100
for ep in range(num_ep):
    obs, info = env.reset(seed=ep)
    steps = 0
    terminated = False

    while steps < 300 and not terminated:
        action = env.action_space.sample() #random policy
        obs, reward, terminated, truncated, info = env.step(action)
        done = truncated or terminated
        steps += 1
     
    if terminated:
        success_lengths.append(steps)
    else:
        failure_lengths.append(steps)

n_success = len(success_lengths)
mean_success = sum(success_lengths) / n_success if n_success > 0 else 0.0
mean_failure = sum(failure_lengths) / len(failure_lengths) if len(failure_lengths) > 0 else 0.0
print("successes:", n_success, "out of ", num_ep)
print("mean success length (steps):", mean_success)
print("mean failure length (steps):", mean_failure)

objc[24605]: Class SDLApplication is implemented in both /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x116eb8890) and /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x1188252c8). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[24605]: Class SDLAppDelegate is implemented in both /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/cv2/.dylibs/libSDL2-2.0.0.dylib (0x116eb88e0) and /Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/pygame/.dylibs/libSDL2-2.0.0.dylib (0x118825318). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[24605]: Class SDLTranslatorResponder is implemented in both /Users/snudurupati/Projects/phy

successes: 0 out of  100
mean success length (steps): 0.0
mean failure length (steps): 300.0


## 2. The Cowboy Policy

In [1]:

def oneline_policy(obs, info):
    goal = np.array([info["goal_pose"][0], info["goal_pose"][1]])
    block = np.array([obs[2], obs[3]])
    margin = 10                                         #margin in pixels, TBD by experimentation
    d = block - goal                                    #points from goal toward block
    norm = np.linalg.norm(d)                            #the length of a vector
    u = d/norm if norm > 0 else np.array([0.0, 0.0])    #same direction, length 1
    action = block + u * margin
    return action


### Same 1000-episode Sim with Cowboy Policy

In [ ]:
import gymnasium as gym
import gym_pusht
import numpy as np 

env = gym.make("gym_pusht/PushT-v0", render_mode="rgb_array")

success_lengths = []   # step counts of episodes that ended in success
failure_lengths = []   # step counts of episodes that hit the 300-step limit

num_ep = 100
for ep in range(num_ep):
    obs, info = env.reset(seed=ep)
    steps = 0
    terminated = False

    while steps < 300 and not terminated:
        action = oneline_policy(obs, info) #defined policy
        obs, reward, terminated, truncated, info = env.step(action)
        done = truncated or terminated
        steps += 1
     
    if terminated:
        success_lengths.append(steps)
    else:
        failure_lengths.append(steps)

n_success = len(success_lengths)
mean_success = sum(success_lengths) / n_success if n_success > 0 else 0.0
mean_failure = sum(failure_lengths) / len(failure_lengths) if len(failure_lengths) > 0 else 0.0
print("successes:", n_success, "out of ", num_ep)
print("mean success length (steps):", mean_success)
print("mean failure length (steps):", mean_failure)

successes: 0 out of  100
mean success length (steps): 0.0
mean failure length (steps): 300.0


## The Cowboy with Human Visibility

In [5]:
env.close()
env = gym.make("gym_pusht/PushT-v0", render_mode="human")

obs, info = env.reset(seed=0)
for steps in range(300):
    action = oneline_policy(obs, info)
    obs, reward, terminated, truncated, info = env.step(action)
    env.render() 
    if steps % 50 == 0:
        print(steps,
              "pusher", np.round(obs[0:2], 1),
              "block", np.round(obs[2:4], 1),
              "action", np.round(action, 1),
              "coverage", round(info["coverage"], 2))
    if terminated:
        break

0 pusher [343.9 295.4] block [241.5 268.5] action [234.  275.1] coverage 0.27
50 pusher [412.1 105.6] block [423.6 181.6] action [431.2 146.2] coverage 0.0
100 pusher [123.6  92.3] block [178.1  37.7] action [145.6  48.2] coverage 0.0
150 pusher [279.7 323.5] block [202.9 319.3] action [232.  334.4] coverage 0.39
200 pusher [338.2  63.8] block [373.7 132.1] action [369.   95.8] coverage 0.0
250 pusher [ 52.  143.1] block [82.4 72.2] action [58.  94.4] coverage 0.0


## 3. Diffusion Policy

```
hf download lerobot/diffusion_pusht README.md config.json train_config.json eval_info.json --local-dir weeks/week-03/model/diffusion_pusht

ls weeks/week-03/modelhf download lerobot/diffusion_pusht model.safetensors --local-dir weeks/week-03/model/diffusion_pusht
```

In [17]:
import shutil, time
from pathlib import Path
import numpy as np
import gymnasium as gym
import gym_pusht
import torch
from lerobot.datasets.lerobot_dataset import LeRobotDataset
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.factory import make_pre_post_processors
from lerobot.envs.utils import preprocess_observation

MIG  = "/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03/model/diffusion_pusht_migrated"
ROOT = "/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03/data/pusht_sim"
DEV  = "mps"
shutil.rmtree(ROOT, ignore_errors=True)          # fresh start

policy = DiffusionPolicy.from_pretrained(MIG)
policy.to(DEV); policy.eval()
pre, post = make_pre_post_processors(
    policy_cfg=policy.config, pretrained_path=MIG,
    preprocessor_overrides={"device_processor": {"device": DEV},
                            "rename_observations_processor": {"rename_map": {}}})

env = gym.make("gym_pusht/PushT-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")

features = {
    "observation.image": {"dtype": "image",   "shape": (96, 96, 3), "names": ["height", "width", "channel"]},
    "observation.state": {"dtype": "float32", "shape": (2,), "names": ["x", "y"]},
    "action":            {"dtype": "float32", "shape": (2,), "names": ["x", "y"]},
    "is_success":        {"dtype": "bool",    "shape": (1,), "names": None},
}
ds = LeRobotDataset.create("pusht_sim", fps=10, features=features,
                           root=ROOT, robot_type="pusht", use_videos=False)

def rollout(make_action, n_episodes, tag):
    for ep in range(n_episodes):
        obs, info = env.reset(seed=ep)
        policy.reset()
        done, n = False, 0
        while not done and n < 300:
            action = make_action(obs, policy, pre, post, DEV)
            obs, r, term, trunc, info = env.step(action)
            ds.add_frame({
                "observation.image": obs["pixels"],
                "observation.state": np.asarray(obs["agent_pos"], dtype=np.float32),
                "action": np.asarray(action, dtype=np.float32),
                "is_success": np.array([bool(info["is_success"])], dtype=bool),
                "task": tag,
            })
            done = term or trunc
            n += 1
        ds.save_episode()
        if (ep + 1) % 50 == 0:
            print(f"{tag}: {ep+1}/{n_episodes} episodes done", flush=True)

t0 = time.time()
rollout(lambda obs, p, pre, post, dev: env.action_space.sample().astype(np.float32),
        1000, "random policy")
ds.finalize()
print(f"RANDOM 1000 done in {time.time()-t0:.0f}s -> {ROOT}")

Loading weights from local directory


/Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/lerobot/datasets/compute_stats.py:154: RuntimeWarning: Converting input from bool to <class 'numpy.uint8'> for compatibility.
  hist, _ = np.histogram(batch[:, i], bins=self._bin_edges[i])
Map: 100%|██████████| 300/300 [00:00<00:00, 4914.62 examples/s]

random policy: 50/1000 episodes done



Map: 100%|██████████| 300/300 [00:00<00:00, 4766.90 examples/s]

random policy: 100/1000 episodes done



Map: 100%|██████████| 300/300 [00:00<00:00, 4984.79 examples/s]

... (31 outputs left out)


Map: 100%|██████████| 300/300 [00:00<00:00, 4922.95 examples/s]

random policy: 950/1000 episodes done



Map: 100%|██████████| 300/300 [00:00<00:00, 4708.17 examples/s]

random policy: 1000/1000 episodes done
RANDOM 1000 done in 436s -> /Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03/data/pusht_sim


In [16]:
import time
import torch
import gymnasium as gym
import gym_pusht
from lerobot.policies.diffusion.modeling_diffusion import DiffusionPolicy
from lerobot.policies.factory import make_pre_post_processors

DEV = "mps" if torch.backends.mps.is_available() else "cpu"
PATH = "/Users/snudurupati/Projects/physical-ai-data-architect/weeks/week-03/model/diffusion_pusht_migrated"   # where the script wrote the output

policy = DiffusionPolicy.from_pretrained(PATH).to(DEV).eval()
pre, post = make_pre_post_processors(policy.config, pretrained_path=PATH)

env = gym.make("gym_pusht/PushT-v0", obs_type="pixels_agent_pos", render_mode="rgb_array")

for ep in range(2):
    t0 = time.time()
    obs, info = env.reset(seed=ep)
    policy.reset()
    steps, done, max_cov = 0, False, 0.0
    while steps < 300 and not done:
        batch = {
            "observation.image": torch.from_numpy(obs["pixels"]).float().div(255)
                .permute(2, 0, 1).unsqueeze(0),
            "observation.state": torch.from_numpy(obs["agent_pos"]).float().unsqueeze(0),
        }
        batch = pre(batch)
        with torch.inference_mode():
            raw = policy.select_action(batch)
        action = post(raw)
        if steps < 3:
            print("raw", raw.min().item(), raw.max().item(),
                  "post", action.min().item(), action.max().item())
        obs, reward, term, trunc, info = env.step(action.squeeze(0).cpu().numpy())
        max_cov = max(max_cov, info["coverage"])
        done = term or trunc or bool(info["is_success"])
        steps += 1
    print(f"ep {ep}: steps={steps} max_cov={max_cov:.3f} "
          f"success={bool(info['is_success'])} {time.time()-t0:.0f}s")

Loading weights from local directory
raw 0.09575355798006058 0.4984034597873688 post 291.26812744140625 385.8516540527344
raw -0.015946153551340103 0.5150646567344666 post 264.1250915527344 390.0086364746094
raw -0.14900852739810944 0.49340930581092834 post 231.79092407226562 384.6056213378906
ep 0: steps=283 max_cov=0.979 success=True 132s
raw -0.08725082129240036 0.05323510989546776 post 246.7980499267578 274.7821350097656
raw -0.058685846626758575 0.18111538887023926 post 253.7393341064453 306.68829345703125
raw -0.01591743528842926 0.24939531087875366 post 264.132080078125 323.7241516113281
ep 1: steps=300 max_cov=0.373 success=False 157s


In [18]:
N_DIFFUSION = 50          # ~100 s/episode → ~85 min; lower if time is tight

ds = LeRobotDataset("pusht_sim", root=ROOT)   # reopen the finalized dataset

def diffusion_action(obs, p, pre, post, dev):
    t = pre(preprocess_observation(obs))       # keys → policy format, batch+device+normalize
    with torch.inference_mode():
        a = p.select_action(t)                 # uses internal 8-action queue between calls
    a = post(a)                                # un-normalize with checkpoint stats
    return np.asarray(a.to("cpu"), dtype=np.float32).squeeze(0)

t0 = time.time()
rollout(diffusion_action, N_DIFFUSION, "diffusion")
ds.finalize()
print(f"DIFFUSION {N_DIFFUSION} done in {time.time()-t0:.0f}s")

/Users/snudurupati/Projects/physical-ai-data-architect/.venv/lib/python3.11/site-packages/lerobot/datasets/compute_stats.py:154: RuntimeWarning: Converting input from bool to <class 'numpy.uint8'> for compatibility.
  hist, _ = np.histogram(batch[:, i], bins=self._bin_edges[i])
Map: 100%|██████████| 210/210 [00:00<00:00, 4820.93 examples/s]

diffusion: 50/50 episodes done
DIFFUSION 50 done in 5309s
